## step 1

prepare dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
df=pd.read_csv(r"C:\Users\Hp\Desktop\binx_mohammadabuhamed\binx_mohammadabuhamed\week3\day5\bank.csv")
df.columns
X = df.drop("deposit", axis=1)
df["deposit"] = df["deposit"].map({"yes": 1, "no": 0})
y = df["deposit"]




In [2]:
df

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,1
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,1
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,1
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,1
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11157,33,blue-collar,single,primary,no,1,yes,no,cellular,20,apr,257,1,-1,0,unknown,0
11158,39,services,married,secondary,no,733,no,no,unknown,16,jun,83,4,-1,0,unknown,0
11159,32,technician,single,secondary,no,29,no,no,cellular,19,aug,156,2,-1,0,unknown,0
11160,43,technician,married,secondary,no,0,no,yes,cellular,8,may,9,2,172,5,failure,0


### see the relations to add new fetures

#### feture 1

##### see the relation between job and the target 

In [3]:
df.groupby("job")["deposit"].mean().sort_values(ascending=False)

job
student          0.747222
retired          0.663239
unemployed       0.565826
management       0.507015
unknown          0.485714
admin.           0.473013
self-employed    0.461728
technician       0.460779
services         0.399783
housemaid        0.397810
entrepreneur     0.375000
blue-collar      0.364198
Name: deposit, dtype: float64

like we see the student have most value and like we see the job is Impact the target



split data

In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    
)

see the real value in just training

In [5]:
datatrain=X_train.copy()
datatrain["deposit"]=y_train
datatrain.groupby('job')["deposit"].mean()

job
admin.           0.479698
blue-collar      0.365100
entrepreneur     0.361446
housemaid        0.394619
management       0.508899
retired          0.645933
self-employed    0.462687
services         0.395412
student          0.719298
technician       0.467361
unemployed       0.531915
unknown          0.535714
Name: deposit, dtype: float64

create verable and make the new feture and put it an job

In [6]:

job=datatrain.groupby('job')["deposit"].mean()
X_train['job']=X_train['job'].map(job)
X_test['job']=X_test['job'].map(job)
print(X_train)

       age       job   marital  education default  balance housing loan  \
3955    28  0.719298    single   tertiary      no     5741      no   no   
11150   34  0.508899   married  secondary      no      355      no   no   
5173    48  0.531915  divorced  secondary      no      201      no   no   
3017    53  0.361446   married   tertiary      no     1961      no   no   
2910    53  0.508899   married   tertiary      no     1624      no   no   
...    ...       ...       ...        ...     ...      ...     ...  ...   
5734    47  0.508899   married   tertiary      no      761     yes   no   
5191    28  0.462687    single   tertiary      no      159      no   no   
5390    35  0.467361   married  secondary      no     1144      no   no   
860     51  0.645933   married   tertiary      no      746      no   no   
7270    30  0.508899    single   tertiary      no        2      no   no   

        contact  day month  duration  campaign  pdays  previous poutcome  
3955   cellular   10   s

#### feture 2

see the diffrances in yes or no

In [18]:
datatrain.groupby('deposit').mean(numeric_only=True)

,age,balance,day,duration,campaign,pdays,previous
deposit,,,,,,,
0,40.891013,1306.733801,16.042065,221.112173,2.831528,35.049076,0.515190
1,41.634060,1758.835860,15.170535,536.466130,2.144008,68.217669,1.141402


like we see here the diffrance between long call and short call

In [19]:
datatrain.groupby("deposit")["duration"].mean()

deposit
0    221.112173
1    536.466130
Name: duration, dtype: float64

In [20]:
datatrain["duration"].describe()

count    8929.000000
mean      370.224549
std       345.949019
min         2.000000
25%       138.000000
50%       253.000000
75%       493.000000
max      3881.000000
Name: duration, dtype: float64

add the feture that see if the call long or not

In [22]:
threshold = X_train["duration"].median()

X_train["long_call"] = (X_train["duration"] > threshold).astype(int)
X_test["long_call"] = (X_test["duration"] > threshold).astype(int)

end of step 1

## oncodeing and scalling

In [23]:
catcols=X_train.select_dtypes(include='object').columns
numcols=X_train.select_dtypes(exclude='object').columns

C:\Users\Hp\AppData\Local\Temp\ipykernel_21004\4131174849.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  catcols=X_train.select_dtypes(include='object').columns


In [25]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
preprocceser=ColumnTransformer(
    transformers=[
        ('num',StandardScaler(),numcols),
        ('cat',OneHotEncoder(handle_unknown='ignore'),catcols)
    ]
    
)
X_trainpre=preprocceser.fit_transform(X_train)
X_testpre=preprocceser.transform(X_test)


## step 2

In [31]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
param_grid={
    "max_depth":[5,7,10,15,20,30,50,100,110]
}


## step 3

In [34]:
grid=GridSearchCV(DecisionTreeClassifier(random_state=42),param_grid,cv=5 ,scoring='accuracy')
grid.fit(X_trainpre, y_train)
print(grid.best_params_)
print(grid.best_score_)
best_model=grid.best_estimator_

{'max_depth': 10}
0.8195770402225839


In [44]:
pred=best_model.predict(X_testpre)
pred2=best_model.predict(X_trainpre)

In [46]:
accuracy_score(y_train,pred2)

0.8779258595587411

In [38]:
from sklearn.metrics import classification_report

In [39]:
print(classification_report(y_test,pred))

              precision    recall  f1-score   support

           0       0.83      0.80      0.81      1166
           1       0.79      0.82      0.80      1067

    accuracy                           0.81      2233
   macro avg       0.81      0.81      0.81      2233
weighted avg       0.81      0.81      0.81      2233



## step 4

the baseline was better the fetures not good

## step 5

the best hyperprametar was 10 depth